# Pipeline Tag Prediction
## Candidate Labeling for Untagged Model Cards

**DATASCI 266: Natural Language Processing with Deep Learning**

UC Berkeley, School of Information

---

This notebook uses the saved TF-IDF + Logistic Regression baseline model to generate
candidate `pipeline_tag` predictions for model cards that are currently **missing**
a `pipeline_tag` on Hugging Face.

The goal here is not final evaluation. It is to build a shortlist for manual review:
the top 10 highest-confidence predictions per class, restricted to the 10 pipeline
tag categories used throughout this project. Once these are manually validated,
they become a new labeled dataset for evaluating the fine-tuned RoBERTa and
ModernBERT models on genuinely unseen, real-world model cards.

It runs in this order:

1. Load saved TF-IDF vectorizer, label encoder, and logistic regression model
2. Load untagged model cards from `librarian-bots/model_cards_with_metadata`
3. Clean text using the same preprocessing as training
4. Generate predictions and class probabilities
5. Select top 10 highest-confidence candidates per class
6. Export candidate list for manual validation


## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import re
import json
import joblib
import warnings

from datasets import load_dataset

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)

print('Libraries loaded ✓')

Libraries loaded ✓


## 1. Load Saved Baseline Artifacts

**Important:** `NB_Baseline_Final.ipynb` fits `tf_idf_vec`, `label_encoder`, and
`lr_model` in memory but does not save them to disk. Before running this notebook,
go back to that notebook and add/run a cell like the one below, then re-run this
notebook.

```python
# Run this in NB_Baseline_Final.ipynb, after the final lr_model is trained on 100% of data
import joblib
joblib.dump(tf_idf_vec, f'{DRIVE_DIR}/tfidf_vectorizer.joblib')
joblib.dump(label_encoder, f'{DRIVE_DIR}/label_encoder.joblib')
joblib.dump(lr_model, f'{DRIVE_DIR}/lr_baseline_model.joblib')
```

Note: the ablation loop retrains `lr_model` fresh at each fraction (25/50/75/100%).
Make sure you save it **after** the 100% run so you're saving the fully-trained
version, not an intermediate one from the ablation.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/266-pipeline-tag-prediction'

tf_idf_vec = joblib.load(f'{DRIVE_DIR}/tfidf_vectorizer.joblib')
label_encoder = joblib.load(f'{DRIVE_DIR}/label_encoder.joblib')
lr_model = joblib.load(f'{DRIVE_DIR}/lr_baseline_model.joblib')

metadata = json.load(open(f'{DRIVE_DIR}/metadata.json'))
label2id, id2label = metadata['label2id'], metadata['id2label']

TOP_10_TAGS = list(label_encoder.classes_)
print("Loaded classes:", TOP_10_TAGS)

Mounted at /content/drive
Loaded classes: ['automatic-speech-recognition', 'image-classification', 'image-text-to-text', 'robotics', 'sentence-similarity', 'text-classification', 'text-generation', 'text-to-image', 'token-classification', 'translation']


## 2. Load the Target Training Distribution

This is used later purely as a reference so you can see, next to each class's top 10
candidates, how large that class was in training. It does not affect the candidate
selection in this notebook (each class independently gets its own top 10), but it's
useful context when you decide how many of each class to keep for your final
100+ card evaluation set.

In [3]:
train_df = pd.read_parquet(f'{DRIVE_DIR}/train_df.parquet')

train_dist = (
    train_df['pipeline_tag']
    .value_counts(normalize=True)
    .rename('train_proportion')
    .reset_index()
    .rename(columns={'index': 'pipeline_tag'})
)
train_dist

,pipeline_tag,train_proportion
0,text-generation,0.482327
1,text-to-image,0.154350
2,image-text-to-text,0.086947
3,text-classification,0.072597
4,translation,0.060641
5,automatic-speech-recognition,0.040885
6,token-classification,0.028778
7,sentence-similarity,0.026598
8,robotics,0.023638
9,image-classification,0.023239


## 3. Load Untagged Model Cards

Pull directly from the same source dataset used for training, filtering for rows
where `pipeline_tag` is missing. This is the candidate pool this notebook selects
from.

In [4]:
raw_dataset = load_dataset('librarian-bots/model_cards_with_metadata', split='train')
full_df = raw_dataset.to_pandas()

print("Total rows:", len(full_df))
full_df[['modelId', 'pipeline_tag']].head()

README.md:   0%|          | 0.00/5.89k [00:00<?, ?B/s]

data/train-00000-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  288MB            

data/train-00000-of-00004.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  290MB            

data/train-00001-of-00004.parquet: downloading bytes:           |  0.00B            

...

data/train-00003-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  288MB            

data/train-00003-of-00004.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/690228 [00:00<?, ? examples/s]

Total rows: 690228


,modelId,pipeline_tag
0,hongjia-kth/lora_model_1205,None
1,texturejc/texture-frames-de-frame,text-classification
2,Brucamian/70-qwen15-55-V-128,text-generation
3,buicuongenzdr17/blockassist-bc-shiny_unseen_caterpillar_1760176561,None
4,komischervogel/psiochecker,None


In [5]:
# Filter for missing pipeline_tag (covers both None and empty string cases)
untagged_df = full_df[
    full_df['pipeline_tag'].isna() | (full_df['pipeline_tag'].astype(str).str.strip() == '')
].copy()

print("Untagged rows:", len(untagged_df))
untagged_df[['modelId', 'pipeline_tag']].head()

Untagged rows: 339247


,modelId,pipeline_tag
0,hongjia-kth/lora_model_1205,None
3,buicuongenzdr17/blockassist-bc-shiny_unseen_caterpillar_1760176561,None
4,komischervogel/psiochecker,None
5,Shiina-Mahiru/Pet-Mischief-Detection,None
8,popekobe7/blockassist-bc-frisky_dense_kiwi_1762358915,None


In [7]:
# Basic sanity filtering: drop rows with no usable card text, mirroring the
# empty-content handling used during original data cleaning
untagged_df['card'] = untagged_df['card'].astype(str)
untagged_df = untagged_df[untagged_df['card'].str.strip().str.len() > 0].reset_index(drop=True)

print("Untagged rows with usable text:", len(untagged_df))

Untagged rows with usable text: 339247


## 4. Clean Text

Same `clean_text` function used in `NB_Baseline_Final.ipynb`, applied here so the
TF-IDF vectorizer sees text in the same format it was fit on. Using a different
cleaning step here would make the model's predictions unreliable, since TF-IDF
features are entirely dependent on exact tokenization.

In [9]:
def clean_text(text: str) -> str:
    """Lowercase, remove punctuation/numbers, collapse whitespace."""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

untagged_df['text_clean'] = untagged_df['card'].apply(clean_text)
untagged_df[['modelId', 'text_clean']].head()

,modelId,text_clean
0,hongjia-kth/lora_model_1205,base model unsloth llama b instruct unsloth bnb bit tags text generation inference transformers unsloth llama trl li...
1,buicuongenzdr17/blockassist-bc-shiny_unseen_caterpillar_1760176561,tags gensyn blockassist gensyn blockassist minecraft shiny unseen caterpillar gensyn blockassist gensyn s blockassis...
2,komischervogel/psiochecker,license other license name illuminator license link license
3,Shiina-Mahiru/Pet-Mischief-Detection,comp computer vision project the pet mischief detector li jianxi d zhang xulu d cheng keyan d task problem definitio...
4,popekobe7/blockassist-bc-frisky_dense_kiwi_1762358915,tags gensyn blockassist gensyn blockassist minecraft frisky dense kiwi gensyn blockassist gensyn s blockassist is a ...


## 5. Generate Predictions and Confidence Scores

Transform the untagged cards with the fitted TF-IDF vectorizer (`transform`, not
`fit_transform`, since we are not refitting on new data) and run them through the
saved logistic regression model. `predict_proba` gives a probability per class,
which is used as a confidence score for ranking candidates within each class.

In [10]:
X_untagged = tf_idf_vec.transform(untagged_df['text_clean'])

pred_probs = lr_model.predict_proba(X_untagged)
pred_ids = np.argmax(pred_probs, axis=1)
pred_confidence = np.max(pred_probs, axis=1)

untagged_df['predicted_tag'] = label_encoder.inverse_transform(pred_ids)
untagged_df['confidence'] = pred_confidence

untagged_df[['modelId', 'predicted_tag', 'confidence']].head(10)

,modelId,predicted_tag,confidence
0,hongjia-kth/lora_model_1205,text-generation,0.975564
1,buicuongenzdr17/blockassist-bc-shiny_unseen_caterpillar_1760176561,text-generation,0.262624
2,komischervogel/psiochecker,text-generation,0.345232
3,Shiina-Mahiru/Pet-Mischief-Detection,image-classification,0.629740
4,popekobe7/blockassist-bc-frisky_dense_kiwi_1762358915,text-generation,0.282482
5,poolkiltzn/blockassist-bc-vigilant_alert_tuna_1758558414,text-generation,0.285229
6,lamhungg/htfhrt,text-generation,0.195364
7,ij/gemma4_e2b_korean_translationese_detector_grpo_lora_test,image-text-to-text,0.923563
8,amethyst9/949985,text-to-image,0.600768
9,lukealonso/MiniMax-M2.7-NVFP4,text-generation,0.768878


In [11]:
# Restrict to the 10 pipeline tag categories used throughout this project.
# (predict_proba only ever returns these 10 classes since that's what the model
# was trained on, so this filter is mostly a safety check / no-op, but keeps
# intent explicit.)
candidates_df = untagged_df[untagged_df['predicted_tag'].isin(TOP_10_TAGS)].copy()

print("Candidate rows:", len(candidates_df))

Candidate rows: 339247


## 6. Distribution Check

Compare the predicted-tag distribution across the untagged pool to the training
distribution. This does not need to match closely (the untagged pool is whatever
it is), but it's useful to see before deciding how many candidates per class you
want in your final validated set.

In [12]:
pred_dist = (
    candidates_df['predicted_tag']
    .value_counts(normalize=True)
    .rename('predicted_proportion')
    .reset_index()
    .rename(columns={'index': 'predicted_tag'})
)

dist_compare = train_dist.merge(
    pred_dist, left_on='pipeline_tag', right_on='predicted_tag', how='outer'
).drop(columns=['predicted_tag']).fillna(0)

dist_compare.sort_values('train_proportion', ascending=False)

,pipeline_tag,train_proportion,predicted_proportion
6,text-generation,0.482327,0.646544
7,text-to-image,0.154350,0.049987
2,image-text-to-text,0.086947,0.062913
5,text-classification,0.072597,0.145313
9,translation,0.060641,0.006635
0,automatic-speech-recognition,0.040885,0.015711
8,token-classification,0.028778,0.004819
4,sentence-similarity,0.026598,0.004876
3,robotics,0.023638,0.014951
1,image-classification,0.023239,0.048251


## 7. Top 10 Highest-Confidence Candidates per Category

For each of the 10 pipeline tags, take the 10 predictions with the highest
confidence score. High confidence doesn't guarantee correctness (the model can be
confidently wrong, especially on classes like `image-text-to-text` that showed
weaker precision in the baseline's classification report), but it's a reasonable
starting point for manual review since it surfaces the clearest cases first.

In [13]:
top10_per_class = (
    candidates_df
    .sort_values('confidence', ascending=False)
    .groupby('predicted_tag')
    .head(10)
    .sort_values(['predicted_tag', 'confidence'], ascending=[True, False])
    .reset_index(drop=True)
)

print("Total shortlisted candidates:", len(top10_per_class))
top10_per_class[['modelId', 'predicted_tag', 'confidence']]

Total shortlisted candidates: 100


,modelId,predicted_tag,confidence
0,Menhaz/distil-whisper-torgo,automatic-speech-recognition,0.999994
1,ruch9265/distil-whisper-torgo,automatic-speech-recognition,0.999994
2,iamgarvit/whisper-small-hi-asr,automatic-speech-recognition,0.999992
3,DelosLogic/accento-v2.0,automatic-speech-recognition,0.999988
4,xb2g/thai-whisper-elderly-healthcare,automatic-speech-recognition,0.999981
...,...,...,...
95,mijuanlo/opus-mt-es-ru-ct2-int8,translation,0.999676
96,mijuanlo/opus-mt-es-de-ct2-int8,translation,0.999638
97,mijuanlo/opus-mt-ca-es-ct2-int8,translation,0.999611
98,mijuanlo/opus-mt-es-ar-ct2-int8,translation,0.999607


In [14]:
# Display each category's top 10 separately for easier manual review
for tag in TOP_10_TAGS:
    print(f"\n{'='*70}")
    print(f"TOP 10 CANDIDATES: {tag}")
    print('='*70)
    subset = top10_per_class[top10_per_class['predicted_tag'] == tag]
    display(subset[['modelId', 'confidence']].reset_index(drop=True))


TOP 10 CANDIDATES: automatic-speech-recognition


,modelId,confidence
0,Menhaz/distil-whisper-torgo,0.999994
1,ruch9265/distil-whisper-torgo,0.999994
2,iamgarvit/whisper-small-hi-asr,0.999992
3,DelosLogic/accento-v2.0,0.999988
4,xb2g/thai-whisper-elderly-healthcare,0.999981
5,MosesJoshuaCoker/SpeechtoText,0.999981
6,vilassn/whisper-small-german,0.999980
7,mlx-community/whisper-large-v3-turbo-asr-fp16,0.999979
8,MosesJoshuaCoker/Krio_fine_tune_novax,0.999979
9,vilassn/whisper-small-german-full-opus7-new,0.999978



TOP 10 CANDIDATES: image-classification


,modelId,confidence
0,phoenix6238/brain-tumor-cnn,0.999961
1,shunya510hi/vit-cifar10-classifier,0.999956
2,Noflowerzzk/Dl_homework,0.999951
3,Aby101/visual-vit,0.999945
4,luciayen/ksl-yolo-cnn-vit-fixed,0.999934
5,0xhalfmoonkid/ritual-models,0.999929
6,Charlie81/genglasses,0.999918
7,KoayGH/real-vs-fake-image-classifier,0.999915
8,muhwira27/skin-lesion-models,0.999909
9,proteus-photos/DINOHash,0.999903



TOP 10 CANDIDATES: image-text-to-text


...

,modelId,confidence
0,kemosato/perfect-ink-drawing-engraving-style,0.999949
1,merlinux2/betty_character,0.999938
2,DavidBaloches/Fairy_Tale_Forest_House,0.999914
3,Kotyar/Flux-Jpfresh-Portrait-LoRA,0.999909
4,akim4ik07/osetin-lora,0.999901
5,avasquat/AvaSquat,0.999829
6,chriswang2025/test-simple-upload,0.999702
7,pyys/Persephone-Flux-2.0-Q8-GGUF,0.999680
8,joyfox/QwenEdit2509_Zootopia,0.999670
9,valiantcat/QwenEdit2509_Zootopia,0.999670



TOP 10 CANDIDATES: token-classification


,modelId,confidence
0,astifer/pii-shield-onnx,0.999967
1,khoirif/xlm-roberta-base-indonesian-ner,0.999954
2,PedroDKE/multilingual-ner-abb,0.999952
3,tlabdev/ner-irish-roberta-base,0.999951
4,rajaadil/finer-ord-ner,0.999948
5,broadfield-dev/bert-mini-ner-pii-training-tuned-12270113-tuned-12271020-tuned-12271353,0.999939
6,barflyman/xlm-roberta-pii-ner-4lang,0.999911
7,jordigonzm/mdeberta-v3-base-multilingual-ner,0.999904
8,PedroDKE/dutch-ner-abb,0.999903
9,sravan1817/xlm-roberta-ner-te-kn,0.999903



TOP 10 CANDIDATES: translation


,modelId,confidence
0,Edrex/opus-mt-en-ja-ft,0.999878
1,mijuanlo/opus-mt-es-en-ct2-int8,0.999842
2,mijuanlo/opus-mt-ca-en-ct2-int8,0.999824
3,Ijaj7/Xetu-assamese-to-english-translator,0.999778
4,OvozifyLabs/nllb-200-uz-en-v1,0.999740
5,mijuanlo/opus-mt-es-ru-ct2-int8,0.999676
6,mijuanlo/opus-mt-es-de-ct2-int8,0.999638
7,mijuanlo/opus-mt-ca-es-ct2-int8,0.999611
8,mijuanlo/opus-mt-es-ar-ct2-int8,0.999607
9,mijuanlo/opus-mt-es-it-ct2-int8,0.999600


## 8. Add Model Card Text Preview for Manual Review

A short text preview makes manual validation faster, since you can sanity-check
the prediction without opening every model card page on Hugging Face.

In [16]:
top10_per_class['text_preview'] = top10_per_class['card'].str.slice(0, 300)

review_df = top10_per_class[[
    'modelId', 'predicted_tag', 'confidence', 'text_preview'
]].reset_index(drop=True)

review_df

,modelId,predicted_tag,confidence,text_preview
0,Menhaz/distil-whisper-torgo,automatic-speech-recognition,0.999994,---\nlanguage: en\nlicense: apache-2.0\nbase_model: openai/whisper-tiny\ndatasets:\n - abnerh/TORGO-database\ntags:...
1,ruch9265/distil-whisper-torgo,automatic-speech-recognition,0.999994,---\nlanguage: en\nlicense: apache-2.0\nbase_model: openai/whisper-tiny\ndatasets:\n - abnerh/TORGO-database\ntags:...
2,iamgarvit/whisper-small-hi-asr,automatic-speech-recognition,0.999992,---\nlanguage:\n- hi\nlicense: apache-2.0\nbase_model: openai/whisper-small\ntags:\n- whisper\n- asr\n- hindi\n- spe...
3,DelosLogic/accento-v2.0,automatic-speech-recognition,0.999988,---\r\nlanguage:\r\n- en\r\ntags:\r\n- whisper\r\n- speech-recognition\r\n- trinidadian-creole\r\n- accent\r\n- asr\...
4,xb2g/thai-whisper-elderly-healthcare,automatic-speech-recognition,0.999981,---\nlanguage:\n- th\ntags:\n- whisper\n- thai\n- asr\n- speech-recognition\n- elderly\n- healthcare\nlicense: apach...
...,...,...,...,...
95,mijuanlo/opus-mt-es-ru-ct2-int8,translation,0.999676,---\nlanguage:\n- es\n- ru\ntags:\n- ctranslate2\n- marian\n- Helsinki-NLP\n- OPUS-MT\n- machine-translation\n- int8...
96,mijuanlo/opus-mt-es-de-ct2-int8,translation,0.999638,---\nlanguage:\n- es\n- de\ntags:\n- ctranslate2\n- marian\n- Helsinki-NLP\n- OPUS-MT\n- machine-translation\n- int8...
97,mijuanlo/opus-mt-ca-es-ct2-int8,translation,0.999611,---\nlanguage:\n- ca\n- es\ntags:\n- ctranslate2\n- marian\n- Helsinki-NLP\n- OPUS-MT\n- machine-translation\n- int8...
98,mijuanlo/opus-mt-es-ar-ct2-int8,translation,0.999607,---\nlanguage:\n- es\n- ar\ntags:\n- ctranslate2\n- marian\n- Helsinki-NLP\n- OPUS-MT\n- machine-translation\n- int8...


## 9. Export Shortlist for Manual Validation

This is the file to work from when manually validating tags. Add a `validated_tag`
column as you review each row (leave it as the `predicted_tag` if you confirm it's
correct, overwrite it if the model got it wrong, or mark it for exclusion if it
doesn't clearly belong in any of the 10 categories).

In [17]:
review_df['validated_tag'] = ''  # fill in during manual review
review_df['include_in_eval_set'] = ''  # e.g. TRUE / FALSE, fill in during review

output_path = f'{DRIVE_DIR}/untagged_candidates_top10_per_class.csv'
review_df.to_csv(output_path, index=False)

print(f"Saved shortlist to {output_path}")
print(f"Rows: {len(review_df)}  |  Categories: {review_df['predicted_tag'].nunique()}")

Saved shortlist to /content/drive/MyDrive/266-pipeline-tag-prediction/untagged_candidates_top10_per_class.csv
Rows: 100  |  Categories: 10


### Next steps

1. Open `untagged_candidates_top10_per_class.csv` and manually review each row,
   ideally by checking the actual model card on Hugging Face for anything the
   300-character preview doesn't make clear.
2. Fill in `validated_tag` (correct the label if needed) and `include_in_eval_set`
   for every row.
3. Since this notebook only pulls 10 per class (100 rows total across the 10
   categories), if you want more than 100 total or need to backfill a class where
   too many candidates got excluded during validation, increase `.head(10)` in
   Section 7 to a larger number (e.g. `.head(15)`) and re-run from Section 7 onward.
4. Once validation is complete, filter to `include_in_eval_set == True` and save
   that as your final evaluation dataset, this becomes the input to the RoBERTa /
   ModernBERT inference notebook.